In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

In [ ]:
PARQUET_DIR = Path("../data/parquet")
DUCKDB_PATH = Path("../ecomerce/dev.duckdb")

tables = {path.stem: pd.read_parquet(path) for path in sorted(PARQUET_DIR.glob("*.parquet"))}
{name: df.shape for name, df in tables.items()}

In [ ]:
orders = tables["orders"].copy()
timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for column in timestamp_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

items = tables["order_items"].copy()
items["price"] = pd.to_numeric(items["price"], errors="coerce")
items["freight_value"] = pd.to_numeric(items["freight_value"], errors="coerce")

payments = tables["order_payments"].copy()
payments["payment_value"] = pd.to_numeric(payments["payment_value"], errors="coerce")

In [ ]:
null_ratio = {name: df.isna().mean().round(4).to_dict() for name, df in tables.items()}
null_ratio

In [ ]:
assert all(not df.empty for df in tables.values())
assert orders["order_id"].is_unique
assert tables["customers"]["customer_id"].is_unique
assert tables["products"]["product_id"].is_unique
assert tables["sellers"]["seller_id"].is_unique
assert set(items["order_id"]).issubset(set(orders["order_id"]))
assert set(payments["order_id"]).issubset(set(orders["order_id"]))
assert (items["price"] >= 0).all()
assert (payments["payment_value"] >= 0).all()
assert orders["order_status"].notna().all()
print("raw validations passed")

In [ ]:
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
fct_orders = con.sql("select * from fct_orders").df()
dim_customers = con.sql("select * from dim_customers").df()
con.close()

assert fct_orders["order_id"].is_unique
assert len(fct_orders) == len(orders)
assert dim_customers["customer_id"].is_unique
assert fct_orders["total_payment_value"].dropna().ge(0).all()
print("marts validations passed", fct_orders.shape)

In [ ]:
fct_orders.groupby("order_status").agg(
    orders=("order_id", "count"),
    revenue=("total_payment_value", "sum"),
    avg_delivery_days=("delivery_days", "mean"),
    avg_review_score=("review_score", "mean"),
).sort_values("orders", ascending=False)